# DocAgent: week 1 on Kaggle

Settings (right panel): **Accelerator: GPU T4 x2**, **Internet: on**.
Replace `YOUR_GITHUB_USER` in the first cell.

In [ ]:
!git clone https://github.com/YOUR_GITHUB_USER/docagent.git
%cd docagent
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## M1: tests (CPU only, a few seconds)

In [ ]:
!python -m pytest -q

## M2: benchmarks
MMLongBench-Doc: ~100 questions from technical document types, 15 PDFs at most.
ViDoRe V3 industrial: 60 English queries over 27 manuals (~5,240 pages; saving the page images takes a few minutes).

In [ ]:
!python scripts/prepare_mmlongbench.py --n 100 --max-docs 15

In [ ]:
!python scripts/prepare_vidore.py --n 60

## M3: own manuals
Fill `data/manuals.csv` first (edit it on GitHub and `git pull`, or write it from a cell).

In [ ]:
!git pull
!python scripts/download_manuals.py

## M4: smoke test, 5 questions per set
Read the `raw` replies before the full run.

In [ ]:
!python eval/run_eval.py --questions data/questions/mmlongbench.jsonl data/questions/vidore_industrial.jsonl --systems B1 B2 --limit 5 --run smoke

In [ ]:
import json
for line in open('results/smoke/predictions.jsonl'):
    r = json.loads(line)
    print(r['system'], r['qid'], 'score', r.get('score'), '|', (r['raw'] or r['skipped'])[:300].replace('\n', ' / '))

## M5: check your own questions

In [ ]:
!python scripts/check_questions.py data/questions/own.jsonl

## M6: full run
Optional: add your W&B key under **Add-ons → Secrets** as `WANDB_API_KEY` and keep `--wandb`.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
    use_wandb = '--wandb'
except Exception:
    use_wandb = ''
print('W&B on' if use_wandb else 'W&B off')

In [ ]:
!python eval/run_eval.py --questions data/questions/mmlongbench.jsonl data/questions/vidore_industrial.jsonl data/questions/own.jsonl --systems B1 B2 --judge llm --run week1 {use_wandb}

## M7: results and error analysis

In [ ]:
from IPython.display import Markdown
Markdown(open('results/week1/summary.md').read())

In [ ]:
!python eval/show_errors.py --run week1 --questions data/questions/*.jsonl -n 10